# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We will demonstrate how to download metadata, inspect available record sets, fields, and columns by their `@id`, and load and process the data.

### Dataset Source
The dataset source is provided via the following Croissant schema URL.

- [FAIR^2 Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Dataset DOI: {getattr(metadata, 'identifier', None)}")
print(f"Dataset License: {getattr(metadata, 'license', None)}")
print(f"Keywords: {getattr(metadata, 'keywords', None)}")

## 2. Data Overview
Review the record sets and fields available in the dataset. All elements are referenced by their `@id` as required for reproducible and precise data access.

For each record set, we will also list its fields and field `@id`s.

In [ ]:
# List all available record sets in the dataset
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")

for recset in record_sets:
    print(f"- Record Set @id: {recset['@id']}")
    print(f"  Name: {recset.get('name', '')}")
    print(f"  Description: {recset.get('description', '')}")
    print(f"  Available fields:")
    for field in recset.get('field', []):
        # field may be a dict (@id or full), or just a @id
        fid = field['@id'] if isinstance(field, dict) else field
        print(f"    - {fid}")
    print()

if not record_sets:
    print("This dataset schema has no RecordSets registered. Please check the Croissant schema for updates or field structure.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 

Below we'll extract the first available record set and preview its content. All field and record set references are always by their `@id`.

In [ ]:
dataframes = {}
if record_sets:
    # Pick the main or first record set
    main_record_set = record_sets[0]['@id']
    print(f"Extracting from main record set: {main_record_set}")
    
    # Load all records (each is a dict of field @id:value)
    records = list(dataset.records(record_set=main_record_set))
    if records:
        df = pd.DataFrame(records)
        dataframes[main_record_set] = df
        print(f"Fields (columns) in this record set:")
        print(df.columns.tolist())
        print('\nPreview of the first 5 records:')
        display(df.head())
    else:
        print("No records found in this record set.")
else:
    print("No record sets present in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by categorical fields.

**Tip:** *Field `@id` references as shown in the data overview are used to index columns.*


In [ ]:
if record_sets and main_record_set in dataframes:
    df = dataframes[main_record_set].copy()
    print("Available fields:")
    print(df.columns.tolist())

    # Heuristically pick a numeric field (e.g. abundance, MW) for analysis.
    # Users should substitute the correct field @id for their analysis.
    numeric_field_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(df[col])]
    if numeric_field_candidates:
        print(f"Numeric fields detected: {numeric_field_candidates}")
        numeric_field = numeric_field_candidates[0]
        print(f"Using field: '{numeric_field}' as the numeric field for EDA.")
    else:
        print("No numeric fields detected. Skipping EDA.")
        numeric_field = None

    if numeric_field:
        # Remove rows with missing values in the numeric field
        df_filtered = df[df[numeric_field].notna()]
        print(f"Total records with non-null '{numeric_field}': {len(df_filtered)}")

        # Filter with threshold - use mean as sample threshold
        threshold = df_filtered[numeric_field].mean()
        print(f"Applying filter: {numeric_field} > {threshold:.2f}")
        df_above = df_filtered[df_filtered[numeric_field] > threshold]
        print(f"Records remaining: {len(df_above)}. Preview:")
        display(df_above.head())

        # Normalize the numeric field (z-score)
        df_above[f"{numeric_field}_normalized"] = (df_above[numeric_field] - df_above[numeric_field].mean()) / df_above[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(df_above[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a likely categorical field (e.g., 'accession', or any with few unique values)
        cat_candidates = [c for c in df.columns if df[c].dtype == object and df[c].nunique() < 15 and c != numeric_field]
        if cat_candidates:
            group_field = cat_candidates[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = df_above.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of '{numeric_field}' by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No extracted data available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and optionally the normalized values or group aggregations.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and main_record_set in dataframes and numeric_field:
    df = dataframes[main_record_set].copy()
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

    # Boxplot by group if available
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to:
- Access a FAIR^2-compliant Croissant dataset using its schema URL
- List available record sets, fields, and their `@id`s for reproducible access
- Load and explore the data using pandas
- Perform basic filtering, normalization, and grouping for exploratory analysis
- Visualize distributions and group means

Users are encouraged to reference the Data Overview outputs to select and use field `@id`s specific to their analysis needs. For more advanced use, explore the full [mlcroissant documentation](https://mlcommons.org/croissant) and refer to the FAIR^2 metadata for complete schema specifications.
